<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/02_Intermediate/L27_Efficient_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L27: Efficient Inference - Batching and Caching

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Intermediate  
**Lesson:** 27 of 30

---

## Learning Objectives

By the end of this lesson, you will:
- Batch multiple inputs for higher throughput
- Use key-value cache for faster autoregressive decoding
- Measure latency and throughput

---

## Concept: Efficient Inference

**Batching**: process multiple sequences together to use GPU efficiently. **KV-cache**: store key/value for previous tokens so we don't recompute them at each step. **Quantization** (L26) also speeds inference. We demonstrate batching and measure time.

---

In [ ]:
!pip install transformers torch -q
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Single vs Batched Inference

---

In [ ]:
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

prompts = ["The capital of France is", "Machine learning is", "In the year 2050,"] * 2

def run_single(prompts, max_new=10):
    start = time.perf_counter()
    for p in prompts:
        inp = tokenizer(p, return_tensors="pt")
        with torch.no_grad():
            model.generate(**inp, max_new_tokens=max_new, pad_token_id=tokenizer.eos_token_id)
    return time.perf_counter() - start

def run_batched(prompts, max_new=10, batch_size=6):
    start = time.perf_counter()
    inp = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        model.generate(**inp, max_new_tokens=max_new, pad_token_id=tokenizer.eos_token_id)
    return time.perf_counter() - start

t_single = run_single(prompts)
t_batch = run_batched(prompts)
print(f"Single: {t_single:.3f}s, Batched: {t_batch:.3f}s")
print(f"Throughput: {len(prompts)/t_batch:.1f} seq/s (batched)")

## Part 2: Use Past Key Values (Cache)

generate() uses KV-cache by default for decoder models. We just ensure we don't disable it.

---

In [ ]:
# HuggingFace generate() uses past_key_values internally for causal LMs.
out = model.generate(**tokenizer(prompts[:1], return_tensors="pt"), max_new_tokens=20, pad_token_id=tokenizer.eos_token_id)
print("Generated (KV-cache used automatically):", tokenizer.decode(out[0], skip_special_tokens=True)[:80])

## Part 3: Padding for Batched Decoding

---

In [ ]:
# Left-padding can help for batch decoding (same length); use tokenizer.padding_side = 'left' for chat.
print("For batched generation: padding_side='left' keeps prompts at end; use attention_mask correctly.")

## Exercises

1. Benchmark different batch sizes (2, 4, 8, 16) and plot throughput vs batch size.
2. Compare inference time with model in FP16 (torch_dtype=torch.float16).
3. Use a simple profiling tool (e.g. PyTorch profiler) to find the bottleneck (attention vs MLP).

---

## Key Takeaways

1. Batching increases throughput; pad sequences and use attention_mask.
2. KV-cache avoids recomputing keys/values for previous tokens in decode.
3. Measure latency (time to first token, total time) and throughput (tokens/s) when optimizing.

---

## Next Lesson

**L28: RAG Systems** – Retrieval-augmented generation.

---